# Timepill 0422 Prototype Detector

## 업로드 방법
1. `0422/` 폴더 전체를 Google Drive 루트에 업로드 (`MyDrive/0422/`)
2. Drive 루트에 아래 폴더들이 있어야 함:
   - `pill.yolov8/` — 실제 알약 labeled 이미지
   - `sample_img/` — 합성용 알약 컷아웃
   - `backgrounds/` — 합성용 배경 이미지
   - `hard_negatives/` — 혼동하기 쉬운 비알약 이미지
   - `aihub이미지데이터/` — AIHub zip 데이터 (Step 3b)

## 실행 순서
Step 1 → Step 2 → Step 3a → Step 3b (AIHub 있을 때) → Step 4 → ...


In [ ]:
import subprocess
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))
else:
    print('GPU not available. In Colab, switch to Runtime > Change runtime type > GPU.')

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout if result.stdout else result.stderr)
except FileNotFoundError:
    print('nvidia-smi not found in this runtime.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%cd /content/drive/MyDrive/0422
!pip install -q ultralytics pillow

In [ ]:
# Step 3b: AIHub 데이터 준비 (optional)
# AIHub 데이터가 없으면 이 셀 건너뛰고, 다음 셀에서 --aihub-root 줄 제거

import zipfile, random, math
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

DRIVE = Path('/content/drive/MyDrive')
BASE  = DRIVE / 'aihub이미지데이터/01.데이터/1.Training'

IMAGES_OUT = Path('/content/aihub_images')
LABELS_OUT = Path('/content/aihub_labels')
IMAGES_OUT.mkdir(exist_ok=True)
LABELS_OUT.mkdir(exist_ok=True)

# 단일경구약제만 사용 (조합은 멀티 bbox라 제외)
IMG_FOLDERS = ['단일경구약제 5000종']

def extract_zips(base: Path, subfolder: str, out_dir: Path, prefix: str):
    src_dir = base / prefix / subfolder
    if not src_dir.exists():
        print(f'  폴더 없음: {src_dir}')
        return
    zips = sorted(src_dir.glob('*.zip'))
    if not zips:
        print(f'  zip 없음: {src_dir}')
        return
    for z_path in zips:
        marker = out_dir / f'.extracted_{z_path.stem}'
        if marker.exists():
            print(f'  이미 추출됨: {z_path.name}')
            continue
        print(f'  추출 중: {z_path.name} → {out_dir}')
        with zipfile.ZipFile(str(z_path), 'r') as z:
            z.extractall(str(out_dir))
        marker.touch()
        print(f'  완료: {z_path.name}')

for folder in IMG_FOLDERS:
    print(f'[이미지] {folder}')
    extract_zips(BASE, folder, IMAGES_OUT, '원천데이터')
    print(f'[라벨]  {folder}')
    extract_zips(BASE, folder, LABELS_OUT, '라벨링데이터')

total_imgs = sum(1 for _ in IMAGES_OUT.rglob('*') if _.is_file())
total_lbls = sum(1 for _ in LABELS_OUT.rglob('*') if _.is_file())
print(f'\n추출 완료 — 이미지: {total_imgs}개 / 라벨: {total_lbls}개')

# AIHub 전처리 (YOLO 형식 변환, 200장 샘플링)
!python preprocess_aihub.py

# ── 결과 카운트 ──
aihub_root = Path('/content/aihub_200')
counts = {}
for split in ('train', 'val'):
    imgs = list((aihub_root / 'images' / split).glob('*'))
    counts[split] = len(imgs)
    print(f'  AIHub {split}: {len(imgs)}장')
print(f'  AIHub 합계: {sum(counts.values())}장')

# ── bbox 포함 샘플 미리보기 ──
def draw_yolo_bbox(img_path: Path, label_dir: Path) -> Image.Image:
    img = Image.open(img_path).convert('RGB')
    lbl = label_dir / f'{img_path.stem}.txt'
    if not lbl.exists():
        return img
    draw = ImageDraw.Draw(img)
    w, h = img.size
    lw = max(2, min(w, h) // 120)
    for line in lbl.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        _, cx, cy, bw, bh = map(float, parts)
        x0 = (cx - bw / 2) * w
        y0 = (cy - bh / 2) * h
        x1 = (cx + bw / 2) * w
        y1 = (cy + bh / 2) * h
        draw.rectangle([x0, y0, x1, y1], outline=(255, 64, 64), width=lw)
    return img

train_imgs = list((aihub_root / 'images' / 'train').glob('*'))
if train_imgs:
    chosen = random.sample(train_imgs, k=min(6, len(train_imgs)))
    cols = 3
    rows = math.ceil(len(chosen) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten().tolist() if hasattr(axes, 'flatten') else [axes]
    lbl_dir = aihub_root / 'labels' / 'train'
    for ax, p in zip(axes, chosen):
        ax.imshow(draw_yolo_bbox(p, lbl_dir))
        ax.set_title(p.name, fontsize=8)
        ax.axis('off')
    for ax in axes[len(chosen):]:
        ax.axis('off')
    fig.suptitle(f'AIHub 샘플 (train {counts["train"]}장 / val {counts["val"]}장) — bbox 표시', fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
!python build_real_prototype_dataset.py \
  --real-root /content/drive/MyDrive \
  --output-root /content/datasets/pill_prototype_0422 \
  --copy-hard-negatives \
  --synthetic-target 100 \
  --val-ratio 0.25 \
  --test-ratio 0 \
  --aihub-root /content/aihub_200
# No AIHub data? Remove the --aihub-root line above.


In [ ]:
import math, random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('/content/datasets/pill_prototype_0422')
SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}

CATEGORIES = [
    ('real_positive',      'Real Positive',      '#4CAF50'),
    ('aihub_positive',     'AIHub Positive',     '#2196F3'),
    ('hard_negative',      'Hard Negative',      '#F44336'),
    ('synthetic_positive', 'Synthetic Positive', '#FF9800'),
]

def classify(name):
    if name.startswith('syn_'):   return 'synthetic_positive'
    if name.startswith('neg_'):   return 'hard_negative'
    if name.startswith('aihub_'): return 'aihub_positive'
    return 'real_positive'

data = {cat: {s: 0 for s in ('train', 'val', 'test')} for cat, *_ in CATEGORIES}
for split in ('train', 'val', 'test'):
    img_dir = ROOT / 'images' / split
    if not img_dir.exists(): continue
    for p in img_dir.iterdir():
        if p.is_file() and p.suffix.lower() in SUFFIXES:
            data[classify(p.name)][split] += 1

# ── 표 출력 ──
print(f'\n{"카테고리":<22} {"Train":>6} {"Val":>6} {"Test":>6} {"합계":>6}')
print('─' * 48)
for cat, label, _ in CATEGORIES:
    t, v, te = data[cat]['train'], data[cat]['val'], data[cat]['test']
    print(f'{label:<22} {t:>6} {v:>6} {te:>6} {t+v+te:>6}')
print('─' * 48)
tr = sum(data[c]['train'] for c, *_ in CATEGORIES)
va = sum(data[c]['val']   for c, *_ in CATEGORIES)
te = sum(data[c]['test']  for c, *_ in CATEGORIES)
print(f'{"TOTAL":<22} {tr:>6} {va:>6} {te:>6} {tr+va+te:>6}')

# ── 누적 막대 차트 ──
splits = ['train', 'val', 'test']
fig, ax = plt.subplots(figsize=(7, 4))
bottoms = np.zeros(3)
for cat, label, color in CATEGORIES:
    vals = np.array([data[cat][s] for s in splits])
    bars = ax.bar(splits, vals, bottom=bottoms, color=color, label=label, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + bar.get_height() / 2,
                    str(val), ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    bottoms += vals
for i, total in enumerate(bottoms):
    ax.text(i, total + 1, str(int(total)), ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('이미지 수')
ax.set_title('데이터셋 구성 (카테고리 × Split)')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, max(bottoms) * 1.15)
plt.tight_layout()
plt.show()


In [ ]:
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

DATASET_ROOT = Path('/content/datasets/pill_prototype_0422')
PREVIEW_SEED = 42
SAMPLES_PER_GROUP = 6
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}

def label_path_for_image(image_path: Path) -> Path:
    split = image_path.parent.name
    return DATASET_ROOT / 'labels' / split / f'{image_path.stem}.txt'

def draw_labeled_image(image_path: Path) -> Image.Image:
    from PIL import ImageOps
    image = ImageOps.exif_transpose(Image.open(image_path)).convert('RGB')
    label_path = label_path_for_image(image_path)
    if not label_path.exists():
        return image

    lines = [line.strip() for line in label_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not lines:
        return image

    draw = ImageDraw.Draw(image)
    width, height = image.size
    line_width = max(2, min(width, height) // 120)
    for line in lines:
        _, center_x, center_y, box_w, box_h = line.split()
        cx = float(center_x) * width
        cy = float(center_y) * height
        bw = float(box_w) * width
        bh = float(box_h) * height
        left = cx - bw / 2
        top = cy - bh / 2
        right = cx + bw / 2
        bottom = cy + bh / 2
        draw.rectangle((left, top, right, bottom), outline=(255, 64, 64), width=line_width)
    return image

def collect_category_samples(dataset_root: Path) -> dict[str, list[Path]]:
    samples = {
        'real_positive': [],
        'aihub_positive': [],
        'hard_negative': [],
        'synthetic_positive': [],
    }
    for split in ('train', 'val', 'test'):
        image_dir = dataset_root / 'images' / split
        if not image_dir.exists():
            continue
        for image_path in sorted(image_dir.iterdir()):
            if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            if image_path.name.startswith('syn_'):
                samples['synthetic_positive'].append(image_path)
            elif image_path.name.startswith('neg_'):
                samples['hard_negative'].append(image_path)
            elif image_path.name.startswith('aihub_'):
                samples['aihub_positive'].append(image_path)
            else:
                samples['real_positive'].append(image_path)
    return samples

def show_random_category(title: str, paths: list[Path], sample_count: int, rng: random.Random) -> None:
    if not paths:
        print(f'{title}: no samples found')
        return

    chosen = rng.sample(paths, k=min(sample_count, len(paths)))
    cols = min(3, len(chosen))
    rows = math.ceil(len(chosen) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
    if hasattr(axes, 'flatten'):
        axes = axes.flatten().tolist()
    else:
        axes = [axes]

    for ax, image_path in zip(axes, chosen):
        ax.imshow(draw_labeled_image(image_path))
        ax.set_title(f'{image_path.parent.name}/{image_path.name}', fontsize=10)
        ax.axis('off')

    for ax in axes[len(chosen):]:
        ax.axis('off')

    fig.suptitle(f'{title} ({len(paths)} total)', fontsize=14)
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

samples = collect_category_samples(DATASET_ROOT)
print({key: len(value) for key, value in samples.items()})

manifest_path = DATASET_ROOT / 'build_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print(json.dumps(manifest.get('counts', {}), indent=2, ensure_ascii=False))

rng = random.Random(PREVIEW_SEED)
show_random_category('Real positives', samples['real_positive'], SAMPLES_PER_GROUP, rng)
show_random_category('AIHub positives', samples['aihub_positive'], SAMPLES_PER_GROUP, rng)
show_random_category('Hard negatives', samples['hard_negative'], SAMPLES_PER_GROUP, rng)
show_random_category('Synthetic positives', samples['synthetic_positive'], SAMPLES_PER_GROUP, rng)


In [ ]:
!python validate_dataset_split.py \
  --dataset-root /content/datasets/pill_prototype_0422 \
  --json-out /content/datasets/pill_prototype_0422/validation_report.json

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count())

In [ ]:
!python train_real_prototype_detector_colab.py \
  --data /content/datasets/pill_prototype_0422/dataset.yaml \
  --project /content/runs \
  --name pill_prototype_0422_v1 \
  --model yolo11n.pt \
  --epochs 120 \
  --batch 32 \
  --patience 30 \
  --close-mosaic 15 \
  --device 0 \
  --export-tflite \
  --drive-export-dir /content/drive/MyDrive/models

In [ ]:
# Step 11: FP / FN 시각화 (best.pt 기준)
# TP=초록, FP=빨강(오탐), FN=파랑(미탐)

import math
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import yaml
from PIL import Image
from ultralytics import YOLO

# ── 설정 ──────────────────────────────────────────────────────────
BEST_PT    = "/content/runs/pill_prototype_0422_v1/weights/best.pt"
DATA_YAML  = "/content/datasets/pill_prototype_0422/dataset.yaml"
SPLIT      = "val"   # "val" or "test"
IOU_THRESH  = 0.5
CONF_THRESH = 0.25
MAX_IMAGES  = 60    # FP/FN 있는 이미지만, 최대 n장
# ──────────────────────────────────────────────────────────────────

def box_iou(b1, b2):
    ix1 = max(b1[0], b2[0]); iy1 = max(b1[1], b2[1])
    ix2 = min(b1[2], b2[2]); iy2 = min(b1[3], b2[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    a1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    a2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    union = a1 + a2 - inter
    return inter/union if union > 0 else 0.0

def yolo_to_xyxy(cx, cy, w, h, iw, ih):
    return [(cx-w/2)*iw, (cy-h/2)*ih, (cx+w/2)*iw, (cy+h/2)*ih]

def load_gt(label_path, iw, ih):
    boxes = []
    if not label_path.exists(): return boxes
    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) == 5:
            _, cx, cy, w, h = map(float, parts)
            boxes.append(yolo_to_xyxy(cx, cy, w, h, iw, ih))
    return boxes

def match(preds, gts, iou_thresh):
    matched_g = set()
    tp_p, fp_p = [], []
    for pi, pb in enumerate(preds):
        best_iou, best_gi = 0, -1
        for gi, gb in enumerate(gts):
            if gi in matched_g: continue
            iou = box_iou(pb, gb)
            if iou > best_iou: best_iou, best_gi = iou, gi
        if best_iou >= iou_thresh:
            tp_p.append(pi); matched_g.add(best_gi)
        else:
            fp_p.append(pi)
    fn_g = [gi for gi in range(len(gts)) if gi not in matched_g]
    return tp_p, fp_p, fn_g

# ── 데이터셋 경로 ─────────────────────────────────────────────────
with open(DATA_YAML, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
dataset_root = Path(cfg.get("path", Path(DATA_YAML).parent))
img_dir = dataset_root / cfg[SPLIT]
lbl_dir = dataset_root / cfg[SPLIT].replace("images", "labels")
image_paths = sorted(p for p in img_dir.rglob("*") if p.suffix.lower() in (".jpg",".jpeg",".png"))
print(f"{SPLIT} 이미지 {len(image_paths)}장")

# ── 추론 ──────────────────────────────────────────────────────────
model = YOLO(BEST_PT)
results = model.predict(
    source=[str(p) for p in image_paths],
    conf=CONF_THRESH, iou=0.45, device="0", verbose=False,
)

# ── FP/FN 분류 ────────────────────────────────────────────────────
error_cases = []
for img_path, result in zip(image_paths, results):
    img = Image.open(img_path)
    iw, ih = img.size
    lbl_path = lbl_dir / (img_path.stem + ".txt")
    gts   = load_gt(lbl_path, iw, ih)
    boxes = result.boxes
    preds = boxes.xyxy.cpu().numpy().tolist() if boxes is not None and len(boxes) else []
    tp_p, fp_p, fn_g = match(preds, gts, IOU_THRESH)
    if fp_p or fn_g:
        error_cases.append((img_path, preds, gts, tp_p, fp_p, fn_g))

print(f"FP/FN 있는 이미지: {len(error_cases)}장")
print(f"  FP(오탐) {sum(len(c[4]) for c in error_cases)}개 | FN(미탐) {sum(len(c[5]) for c in error_cases)}개")

# ── 시각화 ────────────────────────────────────────────────────────
cases = error_cases[:MAX_IMAGES]
ncols = 4
nrows = max(1, math.ceil(len(cases) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*4))
axes = axes.flatten()

COLOR = {"tp": "#00cc44", "fp": "#ff3333", "fn": "#3399ff"}

for ax, (img_path, preds, gts, tp_p, fp_p, fn_g) in zip(axes, cases):
    img = Image.open(img_path)
    ax.imshow(img); ax.axis("off")
    ax.set_title(img_path.name[:28], fontsize=7)
    for pi in tp_p:
        x1,y1,x2,y2 = preds[pi]
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1.5,edgecolor=COLOR["tp"],facecolor="none"))
    for pi in fp_p:
        x1,y1,x2,y2 = preds[pi]
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1.5,edgecolor=COLOR["fp"],facecolor="none"))
        ax.text(x1, y1-4, "FP", color=COLOR["fp"], fontsize=7, fontweight="bold")
    for gi in fn_g:
        x1,y1,x2,y2 = gts[gi]
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1.5,edgecolor=COLOR["fn"],facecolor="none",linestyle="--"))
        ax.text(x1, y1-4, "FN", color=COLOR["fn"], fontsize=7, fontweight="bold")

for ax in axes[len(cases):]:
    ax.axis("off")

from matplotlib.lines import Line2D
legend = [
    Line2D([0],[0],color=COLOR["tp"],lw=2,label="TP (정탐)"),
    Line2D([0],[0],color=COLOR["fp"],lw=2,label="FP (오탐)"),
    Line2D([0],[0],color=COLOR["fn"],lw=2,linestyle="--",label="FN (미탐)"),
]
fig.legend(handles=legend, loc="lower center", ncol=3, fontsize=10, bbox_to_anchor=(0.5,0))
plt.suptitle(f"FP/FN 시각화 — {SPLIT} / conf≥{CONF_THRESH} / IoU≥{IOU_THRESH}", fontsize=12)
plt.tight_layout(rect=[0,0.04,1,1])
plt.show()
